# 12 — Full-data: xác minh và báo cáo

Nạp lại checkpoint, tính lại metric full-test, tạo bảng, biểu đồ và báo cáo riêng cho full-data.

Chọn kernel **NIDS E-GraphSAGE server** trước khi chạy.

In [1]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'research/server').is_dir():
    raise RuntimeError('Hãy mở JupyterLab từ thư mục gốc repository')
THREADS = os.environ.get('NIDS_THREADS', '12')
NUM_WORKERS = os.environ.get('NIDS_NUM_WORKERS', '4')
BUDGET_USD = os.environ.get('NIDS_BUDGET_USD', '6')
HOURLY_PRICE_USD = os.environ.get('NIDS_HOURLY_PRICE_USD', '0')

def full_stage(name):
    command = [sys.executable, '-u', 'research/server/run_full_pipeline.py',
               '--stage', name, '--threads', THREADS, '--num-workers', NUM_WORKERS,
               '--budget-usd', BUDGET_USD, '--hourly-price-usd', HOURLY_PRICE_USD]
    print(' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)

print({'root': str(ROOT), 'threads': THREADS, 'num_workers': NUM_WORKERS,
       'budget_usd': BUDGET_USD, 'hourly_price_usd': HOURLY_PRICE_USD,
       'python': sys.executable,
       'pipeline': 'full-data benchmark-gated'})


{'root': '/workspace/reseach1', 'threads': '12', 'num_workers': '4', 'budget_usd': '6', 'hourly_price_usd': '0', 'python': '/workspace/reseach1/.venv-server/bin/python', 'pipeline': 'full-data benchmark-gated'}


In [2]:
# verify/report/test da chay xong o stage release (xem release_tail.out va
# research/results/full_verification.json). Cell nay chi doc lai ket qua da co.

In [3]:
import pandas as pd
from IPython.display import display, Markdown
display(pd.read_csv(ROOT / 'research/results/full/summary.csv'))
display(Markdown((ROOT / 'research/FULL_DATA_REPORT_VI.md').read_text()))


,dataset,task,model,test_macro_f1_mean,test_macro_f1_std,test_weighted_f1_mean,test_weighted_f1_std,test_accuracy_mean,test_accuracy_std,seconds_fit_and_evaluate_mean,seconds_fit_and_evaluate_std
0,NF-BoT-IoT-v2,binary,edge_mlp,0.911409,0.003841,0.998594,0.000069,0.998460,0.000083,349.634105,6.366506
1,NF-BoT-IoT-v2,binary,sage,0.803625,0.012853,0.996803,0.000121,0.996426,0.000395,545.230859,4.566508
2,NF-BoT-IoT-v2,binary,sage_edge,0.884663,0.027298,0.998086,0.000522,0.997829,0.000651,546.821642,2.669104
3,NF-BoT-IoT-v2,multiclass,edge_mlp,0.814689,0.001882,0.984154,0.000093,0.983930,0.000091,348.307282,6.218145
4,NF-BoT-IoT-v2,multiclass,sage,0.569826,0.020663,0.616899,0.007810,0.629093,0.004636,547.045879,1.497713
5,NF-BoT-IoT-v2,multiclass,sage_edge,0.817143,0.010906,0.978242,0.000907,0.977575,0.000985,545.944896,1.530613
6,NF-CSE-CIC-IDS2018-v2,binary,edge_mlp,0.983403,0.000943,0.993040,0.000394,0.993084,0.000390,163.247895,16.806320
7,NF-CSE-CIC-IDS2018-v2,binary,sage,0.984535,0.001506,0.993504,0.000643,0.993534,0.000651,246.229600,1.819261
8,NF-CSE-CIC-IDS2018-v2,binary,sage_edge,0.985404,0.001647,0.993876,0.000702,0.993911,0.000709,243.816425,3.743749
9,NF-CSE-CIC-IDS2018-v2,multiclass,edge_mlp,0.660685,0.004941,0.745724,0.038500,0.614480,0.047549,170.181160,3.224747


# Báo cáo full-data E-GraphSAGE trên bốn bộ v2

Pipeline đã xử lý toàn bộ 75.987.976 flow, gồm bốn dataset, hai task, ba mô
hình và ba seed, tổng cộng 72 run. Không có bước lấy mẫu bỏ bớt flow train,
validation hoặc test. Batch chỉ là cơ chế đưa toàn bộ cạnh qua GPU.

- Tổng thời gian fit và đánh giá cộng dồn: 5.45 giờ.
- Validator độc lập đã nạp và kiểm tra 72 checkpoint.
- Benchmark trước khi chạy dùng hệ số an toàn 1.35.
- Bảng tổng hợp: `research/results/full/summary.csv`.
- Chỉ số từng lớp: `research/results/full/per_class.csv`.
- Chênh lệch ghép cặp theo cùng seed: `paired_seed_deltas.csv`.
- Cảnh báo lớp có support dưới 1.000: `rare_class_warning.csv`.
- Biểu đồ: `research/results/full/macro_f1_full.png` và `.svg`.

Split được kiểm tra không trùng `flow_group_id`. Tỉ lệ IP validation/test đã
thấy trong train được ghi trong `prepare_manifest.json`; đây là phép đo nguy cơ
rò rỉ danh tính host để giới hạn diễn giải, không phải lỗi chia nhóm flow.
Số nhóm có nhãn mâu thuẫn và số dòng liên quan cũng được báo riêng theo split.

Kết luận khoa học phải dựa trên macro-F1, chỉ số từng lớp và độ lệch chuẩn giữa
ba seed. Bảng paired delta chỉ mang tính mô tả; ba seed không đủ cho khoảng tin
cậy bootstrap ổn định. Các lớp trong `rare_class_warning.csv` không được dùng
để đưa ra kết luận chắc chắn.
